In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler,LabelEncoder 
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [3]:
df = pd.read_csv('used_cars.csv')

In [5]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [18]:
df['accident'].unique()

array(['At least 1 accident or damage reported', 'None reported', nan],
      dtype=object)

In [8]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         4009 non-null   object
 1   model         4009 non-null   object
 2   model_year    4009 non-null   int64 
 3   milage        4009 non-null   object
 4   fuel_type     3839 non-null   object
 5   engine        4009 non-null   object
 6   transmission  4009 non-null   object
 7   ext_col       4009 non-null   object
 8   int_col       4009 non-null   object
 9   accident      3896 non-null   object
 10  clean_title   3413 non-null   object
 11  price         4009 non-null   object
dtypes: int64(1), object(11)
memory usage: 376.0+ KB
None


In [9]:
def remove_letters_keep_numbers(text):
    """
    Takes something like '$25,000' or '10,500 mi' and returns just the number.
    If it's empty/missing, return nothing (NaN).
    """
    if pd.isna(text):
        return np.nan
    numbers_only = re.sub(r"[^\d.]", "", str(text))  # keep digits and dots only
    if numbers_only == "":
        return np.nan
    return float(numbers_only)

In [10]:
def get_engine_size(engine_text):
    """
    Engine column looks like '3.5L V6 24V GDI DOHC'.
    We just want the '3.5' part (engine size in liters).
    """
    if pd.isna(engine_text):
        return np.nan
    match = re.search(r"(\d+\.\d+)L", str(engine_text))
    if match:
        return float(match.group(1))
    return np.nan

In [11]:
def get_cylinder_count(engine_text):
    """
    From '3.5L V6 24V GDI DOHC' we want the '6' in V6.
    """
    if pd.isna(engine_text):
        return np.nan
    match = re.search(r"[VI](\d+)", str(engine_text))
    if match:
        return int(match.group(1))
    return np.nan

In [12]:
def simplify_color(color_text):
    """
    Colors have tons of variations like 'Jet Black Metallic', 'Pearl White'.
    We just check if a common color word is inside the text and use that.
    Anything we don't recognize becomes 'other'.
    """
    color_text = str(color_text).lower()
    common_colors = ["black", "white", "silver", "blue", "red", "gray", "grey"]
    for color in common_colors:
        if color in color_text:
            return "grey" if color == "gray" else color
    return "other"

In [19]:
def had_accident(accident_text):
    if pd.isna(accident_text):
        return 0

    text = accident_text.lower()

    if text == "none reported":
        return 0
    else:
        return 1